In [1]:
import cv2 as cv
import torch
from ultralytics import YOLO
from collections import deque
import time
import numpy as np

In [2]:
def mouse_click(event, x,y, flags,param):
    if event==cv.EVENT_LBUTTONDOWN:
        print( f"({x}, {y})")


In [3]:
print(torch.cuda.is_available())


True


In [4]:
model=YOLO("yolo11n.pt")
cap=cv.VideoCapture("../data/3min.mp4")


In [5]:
colors = {
    "car": (0, 255, 0),
    "bus": (255, 0, 0),
}

In [6]:
#quick remove last item (pop(0))
fps_history=deque(maxlen=30)

In [7]:
x1_roi,y1_roi,x2_roi,y2_roi=250,600,720,850

x: 457, y: 766
x: 601, y: 707
x: 716, y: 757
x: 626, y: 833

x: 286, y: 689
x: 431, y: 646
x: 562, y: 697
x: 426, y: 750

In [8]:
zones_points=[[(460,770),(600,700),(720,760),(630,840)], \
              [(300,700),(430,650),(560,700),(430,750)]]

In [9]:
zone_before=np.array(zones_points[0],dtype=np.int32)
zone_after=np.array(zones_points[1],dtype=np.int32)


In [10]:
def add_layer(img, points, mask_color=(0, 255, 255), alpha=0.3):
    bin_mask = np.zeros((img.shape[:2]), dtype=np.uint8)
    bin_mask = cv.fillPoly(bin_mask, pts=[points], color=1)
    colored_mask = (bin_mask[:, :, np.newaxis] * mask_color).astype(np.uint8)
    return cv.addWeighted(img, 1, colored_mask, alpha, 0)

In [11]:
cap=cv.VideoCapture("../data/3min_day.mp4")
# cv.namedWindow("frame")
# cv.setMouseCallback("frame", mouse_click)
prev_time=time.time()
back_layer=None
frame_idx = 0
last_time = time.time()

while cap.isOpened():
    ret, frame= cap.read()
    if back_layer is None:
        back_layer=add_layer(np.zeros((frame.shape), dtype=np.uint8),zone_before,mask_color=(0, 255, 0))
        back_layer=add_layer(back_layer,zone_after,mask_color=(0, 0, 255))
    # results = model.predict(frame[y1_roi:y2_roi,x1_roi:x2_roi],classes=[2,3,5,7], verbose=False)[0]
    results = model.track(
        frame[y1_roi:y2_roi,x1_roi:x2_roi],
        persist=True, 
        # tracker="bytetrack.yaml", 
        classes=[2,3,5,7], 
        verbose=False, 
        conf=0.6)[0]
    frame=cv.addWeighted(back_layer,1, frame, 1, 0)
    cur_time=time.time()
    fps_history.append(1/(cur_time-prev_time))
    prev_time=time.time()
    avg_fps=sum(fps_history)/len(fps_history)
    cv.putText(frame, f"FPS: {avg_fps:.1f}",(10,30), cv.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)

    if results.boxes and results.boxes.is_track:
        xyxy_all = results.boxes.xyxy.cpu().numpy()
        conf_all = results.boxes.conf.cpu().numpy()
        id_all = results.boxes.id.int().cpu().numpy()
        cls_all=results.boxes.cls.int().cpu().numpy()
        for i in range(len(xyxy_all)):
            x1,y1, x2,y2=map(int, xyxy_all[i])
            x1, x2 = x1 + x1_roi, x2 + x1_roi
            y1, y2 = y1 + y1_roi, y2 + y1_roi
            cv.rectangle(frame,(x1,y1),(x2,y2),(255,0,0),3)
            class_id=int(cls_all[i])
            class_name=model.names[class_id]
            conf=conf_all[i]
            color_ob=colors.get(class_name,(255,255,255))
            track_id = id_all[i]
            ob_label=f"{class_name} #{track_id}  {conf:.2f}"
            cv.putText(frame, ob_label,(x1,y1-8), cv.FONT_HERSHEY_SIMPLEX, 0.5, color_ob, 2)
    cv.imshow("frame",frame)
    if cv.waitKey(1)==ord("q"):
        break
print(avg_fps)
cap.release()
cv.destroyAllWindows()


: 

In [ ]:
from collections import defaultdict

import cv2
import numpy as np

from ultralytics import YOLO

# Load the YOLO26 model
model = YOLO("yolo11n.pt")

# Open the video file
video_path = "../data/3min.mp4"
cap = cv2.VideoCapture(video_path)

# Store the track history
track_history = defaultdict(list)
prev_time=time.time()
back_layer=None
frame_idx = 0
last_time = time.time()
# Loop through the video frames
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()
    if back_layer is None:
        back_layer=add_layer(np.zeros((frame.shape), dtype=np.uint8),zone_before,mask_color=(0, 255, 0))
        back_layer=add_layer(back_layer,zone_after,mask_color=(0, 0, 255))
    if success:
        # Run YOLO26 tracking on the frame, persisting tracks between frames
        result = model.track(
        frame,
        # frame[y1_roi:y2_roi,x1_roi:x2_roi],
        persist=True, 
        tracker="bytetrack.yaml", 
        classes=[2,3,5,7], 
        verbose=False, 
        conf=0.6)[0]
        frame=cv.addWeighted(back_layer,1, frame, 1, 0)
        cur_time=time.time()
        fps_history.append(1/(cur_time-prev_time))
        prev_time=time.time()
        avg_fps=sum(fps_history)/len(fps_history)
        cv.putText(frame, f"FPS: {avg_fps:.1f}",(10,30), cv.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)
        # Get the boxes and track IDs
        if result.boxes and result.boxes.is_track:
            boxes = result.boxes.xyxy.cpu()
            xyxy_all = results.boxes.xyxy.cpu().numpy()
            id_all = results.boxes.id.int().cpu().numpy()
            cls_all=results.boxes.cls.int().cpu().numpy()
            track_ids = result.boxes.id.int().cpu().tolist()
            for i in range(len(xyxy_all)):
                x1,y1, x2,y2=map(int, xyxy_all[i])
                # x1, x2 = x1 + x1_roi, x2 + x1_roi
                # y1, y2 = y1 + y1_roi, y2 + y1_roi
                cv.rectangle(frame,(x1,y1),(x2,y2),(255,0,0),3)
            # Visualize the result on the frame
            # frame = result.plot()

            # # Plot the tracks
            # for box, track_id in zip(boxes, track_ids):
            #     x, y, w, h = box
            #     track = track_history[track_id]
            #     track.append((float(x), float(y)))  # x, y center point
            #     if len(track) > 30:  # retain 30 tracks for 30 frames
            #         track.pop(0)

                # Draw the tracking lines
                # points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                # cv2.polylines(frame, [points], isClosed=False, color=(230, 230, 230), thickness=10)

        # Display the annotated frame
        cv2.imshow("YOLO26 Tracking", frame)

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # Break the loop if the end of the video is reached
        break

# Release the video capture object and close the display window
cap.release()
cv2.destroyAllWindows()

In [32]:
result.boxes 

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([2.], device='cuda:0')
conf: tensor([0.6072], device='cuda:0')
data: tensor([[155.5848, 517.2305, 194.1394, 545.9994,  56.0000,   0.6072,   2.0000]], device='cuda:0')
id: tensor([56.], device='cuda:0')
is_track: True
orig_shape: (1280, 720)
shape: torch.Size([1, 7])
xywh: tensor([[174.8621, 531.6149,  38.5546,  28.7689]], device='cuda:0')
xywhn: tensor([[0.2429, 0.4153, 0.0535, 0.0225]], device='cuda:0')
xyxy: tensor([[155.5848, 517.2305, 194.1394, 545.9994]], device='cuda:0')
xyxyn: tensor([[0.2161, 0.4041, 0.2696, 0.4266]], device='cuda:0')

In [ ]:
results = model.predict(source="../data/3min.mp4", save=True, stream=True)

In [ ]:
for result in results:
    xywh = result.boxes.xywh  # center-x, center-y, width, height
    xywhn = result.boxes.xywhn  # normalized
    xyxy = result.boxes.xyxy  # top-left-x, top-left-y, bottom-right-x, bottom-right-y
    xyxyn = result.boxes.xyxyn  # normalized
    names = [result.names[cls.item()] for cls in result.boxes.cls.int()]  # class name of each box
    confs = result.boxes.conf  # confidence score of each box

In [ ]:
model.names